In [ ]:
#nora
from scipy.optimize import minimize_scalar

# signal strength (>0)
mu_s = 1.0   # CHANGE THIS to see effect

#expected function
def mu_i(beta):
    return beta * b_nominal.values + mu_s * s_i

# Likelihood
def log_likelihood_full(beta):
    expected = mu_i(beta)
    expected = np.clip(expected, 1e-12, None)
    return -np.sum(stats.poisson.logpmf(k_obs, expected))


# Fit β
result = minimize_scalar(log_likelihood_full, bounds=(0.1, 30.0), method='bounded')
beta_hat = result.x

# Expected values at best fit
beta_hat_expected = mu_i(beta_hat)

# Plot
fig, ax = plt.subplots(figsize=(9, 4))

ax.step(s2_bin_edges[:-1], k_obs, where='post', color='black', label='Observed data')

ax.step(s2_bin_edges[:-1], beta_hat_expected, where='post',
        color='blue', linestyle='--',
        label=f'Fit (β={beta_hat:.3f}, μ_s={mu_s})')

ax.set_xscale('log')
ax.set_xlabel('S2 area [PE]')
ax.set_ylabel('Events per bin')
ax.set_xlim(90, 3000)
ax.set_xticks([90, 120, 150, 200, 500, 1000, 3000])
ax.xaxis.set_major_formatter(plt.matplotlib.ticker.ScalarFormatter())

ax.legend()
plt.title("Likelihood Function (Signal + Background)")
plt.tight_layout()
plt.show()

In [ ]:
#nora
# region of interest, energy
roi = (165.3, 271.7)
roi_mask = (s2_bin_centers_log >= roi[0]) & (s2_bin_centers_log <= roi[1])


# define μ_i
def mu_i_roi(beta):
    return beta * b_nominal.values[roi_mask] + mu_s * s_i[roi_mask]

# Likelihood
def log_likelihood_full_roi(beta):
    expected = mu_i_roi(beta)
    expected = np.clip(expected, 1e-12, None)
    return -np.sum(stats.poisson.logpmf(k_obs[roi_mask], expected))

# Fit β
result = minimize_scalar(log_likelihood_full_roi, bounds=(0.1, 30.0), method='bounded')
beta_hat = result.x

# Scan β
beta_values = np.linspace(0.1, 30.0, 500)
log_likes = [-log_likelihood_full_roi(b) for b in beta_values]

# Plot
fig, ax = plt.subplots(figsize=(8, 4))

ax.plot(beta_values, log_likes, color='blue')
ax.axvline(beta_hat, color='red', linestyle='--', label=f'β̂ = {beta_hat:.3f}')

ax.set_xlabel('β (background normalization)')
ax.set_ylabel('log L(β)')
ax.set_title(f'Log-likelihood vs β (μ_s = {mu_s})')

ax.legend()
plt.tight_layout()
plt.show()